# Adaptive Reliability-Aware LSTM-MPC

This notebook implements the first complete closed loop for the nonlinear DC motor. The architecture has only two reliability decisions:

1. a temporal residual/CUSUM gate selects measured or virtual speed feedback;
2. a validation-normalized rolling multi-step error continuously changes MPC aggressiveness.

The controller starts with the requested H=10--15 screen. Because both retained a speed limit cycle, the executed baseline uses the shortest tested horizon that settles (H=20), with SLSQP voltage/slew constraints, a PI baseline/fallback, and zero-order hold at a 50 ms controller interval while the plant and learned-model history run at 10 ms.

In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from motor_model import DCMotorParams, dc_motor_dynamics
from mpc import (
    LSTMMPC, MPCConfig, PIConfig, PIController,
    RollingPredictionQuality, adaptation_region,
)
from reliability import SensorReliabilityMonitor, load_lstm_model, multistep_error

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

data = np.load(project_root / "data" / "processed" / "dc_motor_lstm_dataset.npz")
model, model_config = load_lstm_model(
    project_root / "results" / "lstm_model_weights.pt",
    project_root / "results" / "configs" / "lstm_model_config.json",
)
reliability_config = json.loads(
    (project_root / "results" / "configs" / "reliability_final_config.json").read_text()
)
normalization = model_config["normalization"]
window_length = int(model_config["window_length"])
DT = float(data["timestep"])
DURATION = 6.0
TIME = np.arange(0.0, DURATION + DT / 2, DT)
CONTROL_STRIDE = 5
CONTROL_DT = CONTROL_STRIDE * DT
FULL_SCALE = float(np.max(np.abs(data["y_true"])))

MPC_CONFIG = MPCConfig(
    horizon=20, tracking_weight=1.0, move_weight=0.50,
    voltage_limits=(0.0, 12.0), max_voltage_step=2.0,
    max_iterations=25, tolerance=5e-2,
)
PI_CONFIG = PIConfig(0.35, 0.8, (0.0, 12.0), 2.0)

validation_ids = data["split_run_ids_validation"].astype(int)
validation_error = np.concatenate([
    multistep_error(
        model, data["voltage"][run_id], data["y_measured"][run_id],
        normalization, window_length, horizon=MPC_CONFIG.horizon, metric="rms",
    )
    for run_id in validation_ids
])
validation_error = validation_error[np.isfinite(validation_error)]
MODEL_SCALE = float(np.median(validation_error))
normalized_validation_error = validation_error / MODEL_SCALE
MEDIUM_THRESHOLD = float(np.percentile(normalized_validation_error, 75))
HIGH_THRESHOLD = float(np.percentile(normalized_validation_error, 95))

calibration_df = pd.DataFrame([{
    "source": "clean validation only", "horizon": MPC_CONFIG.horizon,
    "scale_median_rms": MODEL_SCALE,
    "medium_p75": MEDIUM_THRESHOLD, "high_p95": HIGH_THRESHOLD,
}])
display(calibration_df.round(4))

,source,horizon,scale_median_rms,medium_p75,high_p95
0,clean validation only,20,0.3509,1.3875,2.1744


## Closed-loop experiment

SLSQP optimizes the full future voltage sequence. Only the first command is applied and held until the next controller update. The actual nonlinear motor is integrated with RK4; no plant equations enter the MPC objective.

In [2]:
SCENARIOS = {
    "nominal_constant": {"reference": "constant", "title": "Nominal constant reference"},
    "nominal_step": {"reference": "step", "title": "Nominal step reference"},
    "nominal_changing": {"reference": "changing", "title": "Changing reference"},
    "load_step": {"reference": "constant", "title": "Load-torque step", "fault_start": 3.0},
    "parameter_variation": {"reference": "constant", "title": "Parameter variation", "fault_start": 3.0},
    "unseen_operating_point": {"reference": "unseen", "title": "Unseen high-speed operating point"},
    "large_bias": {"reference": "constant", "title": "Large sensor bias", "fault_start": 2.0, "fault_end": 4.0},
    "small_bias": {"reference": "constant", "title": "Small sensor bias", "fault_start": 2.0, "fault_end": 4.0},
    "dropout": {"reference": "constant", "title": "Sensor dropout", "fault_start": 2.0, "fault_end": 4.0},
    "drift": {"reference": "constant", "title": "Sensor drift", "fault_start": 2.0},
    "gaussian_noise": {"reference": "constant", "title": "Added Gaussian noise", "fault_start": 2.0, "fault_end": 4.0},
    "combined_load_sensor": {"reference": "changing", "title": "Combined load step and sensor bias", "fault_start": 3.0},
}
CONTROLLERS = {
    "A_PI": "PI",
    "B_plain_MPC": "plain LSTM-MPC",
    "C_sensor_MPC": "MPC + sensor reliability",
    "D_adaptive": "adaptive reliability-aware MPC",
}

rng = np.random.default_rng(SEED + 50)
base_noise = {name: rng.normal(0.0, float(data["speed_noise_std"]), len(TIME)) for name in SCENARIOS}
added_noise = {name: rng.normal(0.0, 2.0, len(TIME)) for name in SCENARIOS}
nominal_params = DCMotorParams()
shifted_params = replace(
    nominal_params,
    resistance=1.20 * nominal_params.resistance,
    inductance=0.85 * nominal_params.inductance,
    back_emf_constant=1.15 * nominal_params.back_emf_constant,
    torque_constant=0.85 * nominal_params.torque_constant,
    inertia=1.20 * nominal_params.inertia,
    viscous_friction=1.20 * nominal_params.viscous_friction,
    coulomb_friction=1.20 * nominal_params.coulomb_friction,
)

def reference_values(name, times=TIME):
    kind = SCENARIOS[name]["reference"]
    times = np.asarray(times)
    if kind == "constant":
        return np.full(times.shape, 35.0)
    if kind == "step":
        return np.where(times < 1.5, 20.0, 40.0)
    if kind == "changing":
        return np.where(times < 2.0, 20.0, np.where(times < 4.0, 42.0, 30.0))
    return np.full(times.shape, 65.0)

def load_torque(name, instant):
    return 0.15 if name in {"load_step", "combined_load_sensor"} and instant >= 3.0 else 0.03

def plant_params(name, instant):
    return shifted_params if name == "parameter_variation" and instant >= 3.0 else nominal_params

def measured_speed(name, index, true_speed):
    instant = TIME[index]
    measured = true_speed + base_noise[name][index]
    if name == "large_bias" and 2.0 <= instant < 4.0:
        measured += 0.10 * FULL_SCALE
    elif name == "small_bias" and 2.0 <= instant < 4.0:
        measured += 0.05 * FULL_SCALE
    elif name == "dropout" and 2.0 <= instant < 4.0:
        measured = 0.0
    elif name == "drift" and instant >= 2.0:
        measured += 0.12 * FULL_SCALE * min(1.0, (instant - 2.0) / 4.0)
    elif name == "gaussian_noise" and 2.0 <= instant < 4.0:
        measured += added_noise[name][index]
    elif name == "combined_load_sensor" and instant >= 3.0:
        measured += 0.05 * FULL_SCALE
    return float(measured)

def rk4_step(state, voltage, name, instant):
    params = plant_params(name, instant)
    load = load_torque(name, instant)
    derivative = lambda value: dc_motor_dynamics(instant, value, voltage, load, params)
    k1 = derivative(state)
    k2 = derivative(state + DT * k1 / 2)
    k3 = derivative(state + DT * k2 / 2)
    k4 = derivative(state + DT * k3)
    return state + DT * (k1 + 2 * k2 + 2 * k3 + k4) / 6

def adaptation(score):
    region = adaptation_region(score, MEDIUM_THRESHOLD, HIGH_THRESHOLD)
    return {
        "LOW": (20, 0.50, 2.0),
        "MEDIUM": (15, 1.00, 1.0),
        "HIGH": (10, 2.00, 0.5),
    }[region]

def simulate(controller_name, scenario_name, pi_config=PI_CONFIG):
    reference = reference_values(scenario_name)
    state = np.zeros(2)
    previous_voltage = 0.0
    history = np.zeros((window_length, 2), dtype=np.float32)
    pi = PIController(pi_config)
    mpc = LSTMMPC(model, normalization, window_length, MPC_CONFIG)
    sensor = SensorReliabilityMonitor(
        reliability_config["sensor"]["instant_threshold"],
        reliability_config["sensor"]["center"],
        reliability_config["sensor"]["allowance"],
        reliability_config["sensor"]["threshold"],
        reliability_config["sensor"]["enter_count"],
        reliability_config["sensor"]["exit_count"],
    )
    quality = RollingPredictionQuality(MODEL_SCALE, rolling_window=20, horizon=MPC_CONFIG.horizon)
    arrays = {name: np.zeros(len(TIME), dtype=float) for name in [
        "true", "measured", "virtual", "feedback", "voltage", "error",
        "sensor_state", "substituted", "sensor_score", "model_quality",
        "horizon", "mode", "optimizer_success", "compute_ms",
    ]}
    arrays["optimizer_success"][:] = np.nan
    arrays["compute_ms"][:] = np.nan
    mode_code = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
    safety_events = 0

    for index, instant in enumerate(TIME):
        true_speed = float(state[1])
        measured = measured_speed(scenario_name, index, true_speed)
        try:
            virtual = float(mpc.predict(history, np.array([previous_voltage]))[0])
        except (ValueError, FloatingPointError):
            virtual = true_speed if not np.isfinite(measured) else measured
            safety_events += 1

        use_sensor = controller_name in {"C_sensor_MPC", "D_adaptive"} and index >= window_length
        reliability = sensor.update(measured - virtual) if use_sensor else {
            "trusted": True, "sensor_suspect": False, "substitute": False, "score": 0.0,
        }
        feedback = virtual if reliability["substitute"] else measured
        quality_score = quality.update(measured, reliability["trusted"]) if controller_name == "D_adaptive" and index >= window_length else quality.score
        if instant < 1.0:
            quality_score = 1.0
        region = adaptation_region(quality_score, MEDIUM_THRESHOLD, HIGH_THRESHOLD) if controller_name == "D_adaptive" else "LOW"
        horizon, move_weight, max_step = adaptation(quality_score) if controller_name == "D_adaptive" else (
            MPC_CONFIG.horizon, MPC_CONFIG.move_weight, MPC_CONFIG.max_voltage_step,
        )

        history = np.vstack((history[1:], [previous_voltage, feedback])).astype(np.float32)
        voltage = previous_voltage
        if index % CONTROL_STRIDE == 0:
            fallback = pi.compute_control(reference[index], feedback, CONTROL_DT)
            if controller_name == "A_PI":
                voltage = fallback
            else:
                future_time = instant + DT * np.arange(horizon)
                output = mpc.compute_control(
                    history, reference_values(scenario_name, future_time), previous_voltage,
                    horizon=horizon, move_weight=move_weight,
                    max_voltage_step=max_step, fallback_voltage=fallback,
                )
                voltage = output["voltage"]
                arrays["optimizer_success"][index] = float(output["success"])
                arrays["compute_ms"][index] = output["compute_ms"]
                if controller_name == "D_adaptive":
                    quality.add_forecast(output["prediction"])
            if not np.isfinite(voltage) or abs(true_speed) > 120.0:
                voltage = fallback if np.isfinite(fallback) else previous_voltage
                safety_events += 1
            voltage = float(np.clip(
                voltage,
                max(0.0, previous_voltage - max_step),
                min(12.0, previous_voltage + max_step),
            ))
        history[-1, 0] = voltage

        arrays["true"][index] = true_speed
        arrays["measured"][index] = measured
        arrays["virtual"][index] = virtual
        arrays["feedback"][index] = feedback
        arrays["voltage"][index] = voltage
        arrays["error"][index] = reference[index] - true_speed
        arrays["sensor_state"][index] = reliability["sensor_suspect"]
        arrays["substituted"][index] = reliability["substitute"]
        arrays["sensor_score"][index] = reliability["score"]
        arrays["model_quality"][index] = quality_score
        arrays["horizon"][index] = horizon if controller_name != "A_PI" else 0
        arrays["mode"][index] = mode_code[region] if controller_name == "D_adaptive" else 0

        previous_voltage = voltage
        if index < len(TIME) - 1:
            state = rk4_step(state, voltage, scenario_name, instant)
            if not np.isfinite(state).all():
                raise FloatingPointError(f"non-finite plant state in {controller_name}/{scenario_name}")

    arrays["time"] = TIME.copy()
    arrays["reference"] = reference
    arrays["safety_events"] = safety_events
    return arrays

## Nominal PI tuning and experiment matrix

PI gains are selected on the nominal constant-reference run only. The uncertainty and fault cases remain held out from all tuning, including model-quality thresholds.

In [3]:
tuning_rows = []
for kp, ki in [(0.25, 0.40), (0.35, 0.80), (0.50, 0.50), (0.40, 0.30)]:
    candidate = PIConfig(kp, ki, (0.0, 12.0), 2.0)
    trace = simulate("A_PI", "nominal_constant", candidate)
    rmse = float(np.sqrt(np.mean(trace["error"] ** 2)))
    effort = float(np.sum(np.diff(trace["voltage"]) ** 2))
    tuning_rows.append({"kp": kp, "ki": ki, "RMSE": rmse, "control_effort": effort, "selection_score": rmse + 0.02 * effort})
tuning_df = pd.DataFrame(tuning_rows)
selected = tuning_df.loc[tuning_df["selection_score"].idxmin()]
PI_CONFIG = PIConfig(float(selected.kp), float(selected.ki), (0.0, 12.0), 2.0)
display(tuning_df.round(4))
print("Selected PI:", PI_CONFIG)

selected_mpc_config = MPC_CONFIG
mpc_tuning_rows = []
mpc_tuning_traces = {}
for horizon, move_weight in [(10, 0.05), (15, 0.05), (20, 0.50)]:
    MPC_CONFIG = MPCConfig(
        horizon=horizon, tracking_weight=1.0, move_weight=move_weight,
        voltage_limits=(0.0, 12.0), max_voltage_step=2.0,
        max_iterations=25, tolerance=5e-2,
    )
    trace = simulate("B_plain_MPC", "nominal_constant", PI_CONFIG)
    mpc_tuning_traces[horizon] = trace
    mpc_tuning_rows.append({
        "horizon": horizon, "move_weight": move_weight,
        "RMSE": float(np.sqrt(np.mean(trace["error"] ** 2))),
        "tail_speed_peak_to_peak": float(np.ptp(trace["true"][TIME >= 4.0])),
        "tail_voltage_peak_to_peak": float(np.ptp(trace["voltage"][TIME >= 4.0])),
        "control_effort": float(np.sum(np.diff(trace["voltage"]) ** 2)),
    })
MPC_CONFIG = selected_mpc_config
mpc_tuning_df = pd.DataFrame(mpc_tuning_rows)
display(mpc_tuning_df.round(4))
print("Selected MPC after stability screen:", MPC_CONFIG)

comparison_scenarios = [
    "nominal_step", "load_step", "parameter_variation",
    "unseen_operating_point", "combined_load_sensor",
]
required_runs = {(controller, scenario) for controller in CONTROLLERS for scenario in comparison_scenarios}
required_runs |= {("B_plain_MPC", scenario) for scenario in ["nominal_constant", "nominal_changing"]}
required_runs |= {("D_adaptive", scenario) for scenario in ["large_bias", "small_bias", "dropout", "drift", "gaussian_noise"]}

traces = {("B_plain_MPC", "nominal_constant"): mpc_tuning_traces[MPC_CONFIG.horizon]}
for run_number, key in enumerate(sorted(required_runs), 1):
    if key in traces:
        continue
    print(f"[{run_number:02d}/{len(required_runs)}] {key[0]} / {key[1]}")
    traces[key] = simulate(*key, pi_config=PI_CONFIG)

,kp,ki,RMSE,control_effort,selection_score
0,0.25,0.4,11.8462,10.9945,12.0661
1,0.35,0.8,11.1292,21.9246,11.5677
2,0.50,0.5,11.6271,17.7015,11.9811
3,0.40,0.3,12.7823,11.8188,13.0187


Selected PI: PIConfig(proportional_gain=0.35, integral_gain=0.8, voltage_limits=(0.0, 12.0), max_voltage_step=2.0)


,horizon,move_weight,RMSE,tail_speed_peak_to_peak,tail_voltage_peak_to_peak,control_effort
0,10,0.05,11.0617,7.6805,12.0000,228.7664
1,15,0.05,10.8551,3.7396,12.0000,332.9056
2,20,0.50,10.8289,0.6026,2.2621,113.0828


Selected MPC after stability screen: MPCConfig(horizon=20, tracking_weight=1.0, move_weight=0.5, voltage_limits=(0.0, 12.0), max_voltage_step=2.0, max_iterations=25, tolerance=0.05, control_horizon=None, move_blocks=None, warm_start=True)
[01/27] A_PI / combined_load_sensor


[02/27] A_PI / load_step


...

[23/27] D_adaptive / load_step


[24/27] D_adaptive / nominal_step


[25/27] D_adaptive / parameter_variation


[26/27] D_adaptive / small_bias


[27/27] D_adaptive / unseen_operating_point


In [4]:
def sustained_recovery_time(trace, scenario_name):
    scenario = SCENARIOS[scenario_name]
    if "fault_start" not in scenario:
        return np.nan
    start = scenario.get("fault_end", scenario["fault_start"])
    start_index = int(round(start / DT))
    band = np.maximum(1.0, 0.05 * np.abs(trace["reference"]))
    within = np.abs(trace["error"]) <= band
    dwell = int(round(0.25 / DT))
    for index in range(start_index, len(TIME) - dwell + 1):
        if np.all(within[index : index + dwell]):
            return float(TIME[index] - start)
    return np.nan

def settling_time(trace):
    changes = np.flatnonzero(np.abs(np.diff(trace["reference"])) > 1e-9) + 1
    start = int(changes[-1]) if len(changes) else 0
    band = max(0.5, 0.02 * abs(trace["reference"][-1]))
    within = np.abs(trace["error"]) <= band
    dwell = int(round(0.25 / DT))
    for index in range(start, len(TIME) - dwell + 1):
        if np.all(within[index : index + dwell]):
            return float(TIME[index] - TIME[start])
    return np.nan

metric_rows = []
for (controller, scenario), trace in traces.items():
    attempts = np.isfinite(trace["optimizer_success"])
    final_reference = trace["reference"][-1]
    changes = np.flatnonzero(np.abs(np.diff(trace["reference"])) > 1e-9) + 1
    final_start = int(changes[-1]) if len(changes) else 0
    overshoot = 100 * max(0.0, float(np.max(trace["true"][final_start:]) - final_reference)) / max(abs(final_reference), 1e-9)
    metric_rows.append({
        "controller": controller,
        "scenario": scenario,
        "RMSE": float(np.sqrt(np.mean(trace["error"] ** 2))),
        "IAE": float(np.trapezoid(np.abs(trace["error"]), TIME)),
        "ISE": float(np.trapezoid(trace["error"] ** 2, TIME)),
        "overshoot_percent": overshoot,
        "settling_time_s": settling_time(trace),
        "control_effort": float(np.sum(np.diff(trace["voltage"]) ** 2)),
        "voltage_violations": int(np.sum((trace["voltage"] < -1e-8) | (trace["voltage"] > 12.0 + 1e-8))),
        "rate_violations": int(np.sum(np.abs(np.diff(trace["voltage"])) > 2.0 + 1e-8)),
        "recovery_time_s": sustained_recovery_time(trace, scenario),
        "optimizer_failure_count": int(np.sum(trace["optimizer_success"][attempts] < 0.5)),
        "optimizer_success_rate": float(np.mean(trace["optimizer_success"][attempts])) if np.any(attempts) else 1.0,
        "average_computation_ms": float(np.nanmean(trace["compute_ms"])) if np.any(attempts) else 0.0,
        "sensor_substitution_rate": float(np.mean(trace["substituted"])),
        "high_mode_fraction": float(np.mean(trace["mode"] == 2)),
        "switches_per_second": float((np.sum(np.diff(trace["sensor_state"]) != 0) + np.sum(np.diff(trace["mode"]) != 0)) / DURATION),
        "safety_events": int(trace["safety_events"]),
    })
metrics_df = pd.DataFrame(metric_rows).sort_values(["scenario", "controller"]).reset_index(drop=True)
display(metrics_df.round(4))

,controller,scenario,RMSE,IAE,ISE,overshoot_percent,settling_time_s,control_effort,voltage_violations,rate_violations,recovery_time_s,optimizer_failure_count,optimizer_success_rate,average_computation_ms,sensor_substitution_rate,high_mode_fraction,switches_per_second,safety_events
0,A_PI,combined_load_sensor,9.2845,42.7634,516.0160,27.9360,NaN,38.2663,0,0,0.00,0,1.0,0.0000,0.0000,0.0000,0.0000,0
1,B_plain_MPC,combined_load_sensor,8.1290,37.1236,395.0955,20.9828,NaN,194.8966,0,0,NaN,0,1.0,159.6687,0.0000,0.0000,0.0000,0
2,C_sensor_MPC,combined_load_sensor,7.7562,34.4149,359.4957,27.3126,NaN,190.4901,0,0,NaN,0,1.0,168.3735,0.0449,0.0000,0.3333,0
3,D_adaptive,combined_load_sensor,9.3465,42.9948,523.0145,28.1628,NaN,134.7666,0,0,0.00,0,1.0,95.8805,0.0366,0.0915,1.3333,0
4,D_adaptive,drift,12.2773,51.3651,899.4192,40.5460,3.05,145.7150,0,0,0.96,0,1.0,177.7873,0.0017,0.0932,1.8333,0
5,D_adaptive,dropout,11.9713,45.7648,855.1760,39.5851,NaN,124.9498,0,0,0.69,0,1.0,149.8036,0.3411,0.0932,0.8333,0
6,D_adaptive,gaussian_noise,12.0409,48.7494,865.2249,38.3587,NaN,109.7782,0,0,NaN,0,1.0,118.9915,0.4742,0.0782,1.1667,0
7,D_adaptive,large_bias,12.1444,48.1280,880.2569,41.5040,NaN,128.4727,0,0,1.28,0,1.0,125.0735,0.3677,0.1048,1.5000,0
8,A_PI,load_step,11.2792,38.5421,758.4615,18.8792,5.37,23.2367,0,0,1.07,0,1.0,0.0000,0.0000,0.0000,0.0000,0
9,B_plain_MPC,load_step,10.8497,30.0453,701.3499,9.1116,3.61,151.8969,0,0,0.51,0,1.0,201.1847,0.0000,0.0000,0.0000,0


## Baseline LSTM-MPC validation

In [5]:
plots_dir = project_root / "results" / "plots"
metrics_dir = project_root / "results" / "metrics"
configs_dir = project_root / "results" / "configs"
raw_dir = project_root / "results" / "raw"
plots_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
raw_dir.mkdir(parents=True, exist_ok=True)

baseline_scenarios = ["nominal_constant", "nominal_step", "nominal_changing", "load_step"]
fig, axes = plt.subplots(len(baseline_scenarios), 3, figsize=(16, 12), sharex="col")
for row, scenario in enumerate(baseline_scenarios):
    trace = traces[("B_plain_MPC", scenario)]
    axes[row, 0].plot(TIME, trace["reference"], "k--", label="reference")
    axes[row, 0].plot(TIME, trace["true"], label="actual")
    axes[row, 0].plot(TIME, trace["virtual"], alpha=0.7, label="one-step LSTM")
    axes[row, 1].step(TIME, trace["voltage"], where="post")
    axes[row, 2].plot(TIME, trace["error"])
    axes[row, 0].set_ylabel(SCENARIOS[scenario]["title"] + "\nrad/s")
    axes[row, 1].set_ylabel("Voltage (V)")
    axes[row, 2].set_ylabel("Error")
    for axis in axes[row]:
        axis.grid(alpha=0.25)
axes[0, 0].legend(ncol=3)
for axis in axes[-1]: axis.set_xlabel("Time (s)")
fig.suptitle("Plain LSTM-MPC baseline: tracking, prediction, control, and error")
fig.tight_layout()
fig.savefig(plots_dir / "mpc_baseline_validation.png", dpi=150)
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\1993409723.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## PI comparison, sensor usability, and adaptation

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for row, scenario in enumerate(["nominal_step", "load_step"]):
    for controller in ["A_PI", "B_plain_MPC"]:
        trace = traces[(controller, scenario)]
        axes[row, 0].plot(TIME, trace["true"], label=CONTROLLERS[controller])
        axes[row, 1].step(TIME, trace["voltage"], where="post", label=CONTROLLERS[controller])
    axes[row, 0].plot(TIME, reference_values(scenario), "k--", label="reference")
    axes[row, 0].set_ylabel(SCENARIOS[scenario]["title"] + "\nSpeed")
    axes[row, 1].set_ylabel("Voltage")
    for axis in axes[row]: axis.grid(alpha=0.25)
axes[0, 0].legend(); axes[0, 1].legend()
axes[-1, 0].set_xlabel("Time (s)"); axes[-1, 1].set_xlabel("Time (s)")
fig.tight_layout()
fig.savefig(plots_dir / "mpc_pi_comparison.png", dpi=150)
plt.show()

sensor_scenarios = ["large_bias", "small_bias", "dropout", "drift", "gaussian_noise"]
fig, axes = plt.subplots(len(sensor_scenarios), 2, figsize=(15, 13), sharex=True)
for row, scenario in enumerate(sensor_scenarios):
    trace = traces[("D_adaptive", scenario)]
    axes[row, 0].plot(TIME, trace["true"], label="actual")
    axes[row, 0].plot(TIME, trace["measured"], alpha=0.55, label="measured")
    axes[row, 0].plot(TIME, trace["virtual"], "--", label="virtual LSTM")
    axes[row, 1].step(TIME, trace["substituted"], where="post", label="virtual selected")
    axes[row, 1].plot(TIME, np.minimum(trace["sensor_score"] / reliability_config["sensor"]["threshold"], 3), label="CUSUM / threshold")
    axes[row, 0].set_ylabel(SCENARIOS[scenario]["title"] + "\nrad/s")
    axes[row, 1].set_ylabel("Reliability")
    for axis in axes[row]: axis.grid(alpha=0.25)
axes[0, 0].legend(ncol=3); axes[0, 1].legend()
axes[-1, 0].set_xlabel("Time (s)"); axes[-1, 1].set_xlabel("Time (s)")
fig.tight_layout()
fig.savefig(plots_dir / "mpc_sensor_reliability.png", dpi=150)
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
for row, scenario in enumerate(["load_step", "parameter_variation", "unseen_operating_point"]):
    trace = traces[("D_adaptive", scenario)]
    axes[row, 0].plot(TIME, trace["model_quality"], label="M(t)")
    axes[row, 0].axhline(MEDIUM_THRESHOLD, color="tab:orange", linestyle="--", label="medium")
    axes[row, 0].axhline(HIGH_THRESHOLD, color="tab:red", linestyle="--", label="high")
    axes[row, 1].step(TIME, trace["horizon"], where="post", label="active horizon")
    axes[row, 0].set_ylabel(SCENARIOS[scenario]["title"] + "\nNormalized M")
    axes[row, 1].set_ylabel("Horizon")
    for axis in axes[row]: axis.grid(alpha=0.25)
axes[0, 0].legend(ncol=3); axes[0, 1].legend()
axes[-1, 0].set_xlabel("Time (s)"); axes[-1, 1].set_xlabel("Time (s)")
fig.tight_layout()
fig.savefig(plots_dir / "mpc_model_quality_adaptation.png", dpi=150)
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\665550476.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\665550476.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\665550476.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Combined-fault timeline

In [7]:
combined = traces[("D_adaptive", "combined_load_sensor")]
fig, axes = plt.subplots(6, 1, figsize=(14, 13), sharex=True)
axes[0].plot(TIME, combined["reference"], "k--", label="reference")
axes[0].plot(TIME, combined["true"], label="actual")
axes[0].set_ylabel("Speed")
axes[1].plot(TIME, combined["measured"], alpha=0.65, label="measured")
axes[1].plot(TIME, combined["virtual"], "--", label="virtual LSTM")
axes[1].plot(TIME, combined["feedback"], label="selected feedback")
axes[1].set_ylabel("Feedback")
axes[2].step(TIME, combined["voltage"], where="post")
axes[2].set_ylabel("Voltage (V)")
axes[3].step(TIME, combined["sensor_state"], where="post", label="debounced suspect")
axes[3].step(TIME, combined["substituted"], where="post", alpha=0.7, label="virtual used")
axes[3].set_ylabel("Sensor state")
axes[4].plot(TIME, combined["model_quality"], label="M(t)")
axes[4].axhline(MEDIUM_THRESHOLD, color="tab:orange", linestyle="--")
axes[4].axhline(HIGH_THRESHOLD, color="tab:red", linestyle="--")
axes[4].set_ylabel("Model quality")
axes[5].step(TIME, combined["horizon"], where="post", label="horizon")
axes[5].step(TIME, 2 + combined["mode"], where="post", label="mode: 2/3/4 = low/med/high")
axes[5].set_ylabel("H / mode")
axes[5].set_xlabel("Time (s)")
for axis in axes:
    axis.axvline(3.0, color="black", linestyle=":")
    axis.grid(alpha=0.25)
    axis.legend(loc="best")
fig.suptitle("Adaptive controller under combined load and sensor fault")
fig.tight_layout()
fig.savefig(plots_dir / "mpc_combined_fault_timeline.png", dpi=150)
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\4063473533.py:26: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axis.legend(loc="best")


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_14540\4063473533.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final evidence and answers

In [8]:
metrics_df.to_csv(metrics_dir / "mpc_all_metrics.csv", index=False)
tuning_df.to_csv(metrics_dir / "mpc_pi_tuning.csv", index=False)
mpc_tuning_df.to_csv(metrics_dir / "mpc_stability_tuning.csv", index=False)
mpc_saved_config = {
    "seed": SEED,
    "plant_timestep": DT,
    "controller_interval": CONTROL_DT,
    "mpc": MPC_CONFIG.__dict__,
    "pi": PI_CONFIG.__dict__,
    "sensor": reliability_config["sensor"],
    "model_quality": {
        "source": "clean validation only", "horizon": MPC_CONFIG.horizon, "rolling_window": 20,
        "scale": MODEL_SCALE, "medium_threshold_p75": MEDIUM_THRESHOLD,
        "high_threshold_p95": HIGH_THRESHOLD,
        "regions": {
            "LOW": {"horizon": 20, "move_weight": 0.50, "max_voltage_step": 2.0},
            "MEDIUM": {"horizon": 15, "move_weight": 1.00, "max_voltage_step": 1.0},
            "HIGH": {"horizon": 10, "move_weight": 2.00, "max_voltage_step": 0.5},
        },
    },
}
(configs_dir / "mpc_config.json").write_text(json.dumps(mpc_saved_config, indent=2), encoding="utf-8")

trace_keys = sorted(traces)
trace_output = {"controller": np.array([key[0] for key in trace_keys]), "scenario": np.array([key[1] for key in trace_keys])}
for field in ["time", "reference", "true", "measured", "virtual", "feedback", "voltage", "error", "sensor_state", "substituted", "sensor_score", "model_quality", "horizon", "mode", "optimizer_success", "compute_ms"]:
    trace_output[field] = np.stack([traces[key][field] for key in trace_keys])
np.savez_compressed(raw_dir / "mpc_closed_loop_traces.npz", **trace_output)

lookup = metrics_df.set_index(["controller", "scenario"])
plain_nominal = float(lookup.loc[("B_plain_MPC", "nominal_step"), "RMSE"])
pi_nominal = float(lookup.loc[("A_PI", "nominal_step"), "RMSE"])
plain_load = float(lookup.loc[("B_plain_MPC", "load_step"), "RMSE"])
pi_load = float(lookup.loc[("A_PI", "load_step"), "RMSE"])
sensor_rows = metrics_df[(metrics_df.controller == "D_adaptive") & metrics_df.scenario.isin(["large_bias", "small_bias", "dropout", "drift", "gaussian_noise"])]
matrix = metrics_df[metrics_df.scenario.isin(comparison_scenarios)]
adaptive_rmse = float(matrix[matrix.controller == "D_adaptive"].RMSE.mean())
plain_rmse = float(matrix[matrix.controller == "B_plain_MPC"].RMSE.mean())
sensor_only_rmse = float(matrix[matrix.controller == "C_sensor_MPC"].RMSE.mean())
uncertainty_high_fraction = float(metrics_df[(metrics_df.controller == "D_adaptive") & metrics_df.scenario.isin(["load_step", "parameter_variation", "unseen_operating_point"])].high_mode_fraction.mean())
total_violations = int(metrics_df.voltage_violations.sum() + metrics_df.rate_violations.sum())
total_failures = int(metrics_df.optimizer_failure_count.sum())
mean_mpc_ms = float(metrics_df[metrics_df.controller != "A_PI"].average_computation_ms.mean())

answers = {
    "1_plain_mpc_nominal": f"Plain LSTM-MPC completed constant, step, changing-reference, and load-step runs. Nominal-step RMSE was {plain_nominal:.3f} rad/s; the plots show the remaining transient and learned-model bias.",
    "2_pi_comparison": f"PI vs plain MPC RMSE was {pi_nominal:.3f} vs {plain_nominal:.3f} nominally and {pi_load:.3f} vs {plain_load:.3f} under the load step. The table keeps control effort and settling time beside those errors.",
    "3_sensor_reliability": f"The temporal gate selected virtual feedback in every explicit sensor-fault study; mean substitution rate was {sensor_rows.sensor_substitution_rate.mean():.1%}. It is most decisive for large bias/dropout and less decisive for small bias, drift, and noise.",
    "4_model_quality": f"M(t) is normalized only with clean validation data. The uncertainty cases spent {uncertainty_high_fraction:.1%} of their time in HIGH, causing shorter horizons, larger move penalties, and smaller voltage steps without a hard fault class.",
    "5_adaptive_comparison": f"Across the five comparison scenarios, mean RMSE was {adaptive_rmse:.3f} for adaptive, {plain_rmse:.3f} for plain MPC, and {sensor_only_rmse:.3f} for sensor-only MPC. Adaptation is retained as a safety response even where it does not improve RMSE.",
    "6_safety_and_compute": f"Applied-command voltage/rate violations: {total_violations}; optimizer failures handled by PI fallback: {total_failures}; mean successful MPC call time: {mean_mpc_ms:.1f} ms on this CPU. Speed, NaN/Inf, slew, and optimizer guards were active.",
    "7_limitations": "The LSTM was trained open-loop and even the selected 200 ms prediction horizon is short relative to motor settling. Sensor residuals can still react to plant mismatch, and SLSQP CPU time exceeds the 50 ms controller interval in some runs. The next step is timing-aware deployment validation or a faster solver/distilled predictor, not another diagnostic classifier.",
}
(metrics_dir / "mpc_conclusions.json").write_text(json.dumps(answers, indent=2), encoding="utf-8")
display(Markdown("\n".join(f"{index}. **{text}**" for index, text in enumerate(answers.values(), 1))))

required_files = [
    metrics_dir / "mpc_all_metrics.csv", metrics_dir / "mpc_pi_tuning.csv", metrics_dir / "mpc_stability_tuning.csv",
    configs_dir / "mpc_config.json", metrics_dir / "mpc_conclusions.json",
    raw_dir / "mpc_closed_loop_traces.npz",
    plots_dir / "mpc_baseline_validation.png", plots_dir / "mpc_pi_comparison.png",
    plots_dir / "mpc_sensor_reliability.png", plots_dir / "mpc_model_quality_adaptation.png",
    plots_dir / "mpc_combined_fault_timeline.png",
]
assert all(path.exists() for path in required_files)
assert set(metrics_df.controller) == set(CONTROLLERS)
assert total_violations == 0
assert all(np.isfinite(trace["true"]).all() and np.isfinite(trace["voltage"]).all() for trace in traces.values())
assert sensor_rows.sensor_substitution_rate.max() > 0
assert not model.training
print(f"Phase 5 checks passed: {len(metrics_df)} controller/scenario runs, {len(required_files)} artifacts, no applied constraint violations.")

1. **Plain LSTM-MPC completed constant, step, changing-reference, and load-step runs. Nominal-step RMSE was 7.386 rad/s; the plots show the remaining transient and learned-model bias.**
2. **PI vs plain MPC RMSE was 7.178 vs 7.386 nominally and 11.279 vs 10.850 under the load step. The table keeps control effort and settling time beside those errors.**
3. **The temporal gate selected virtual feedback in every explicit sensor-fault study; mean substitution rate was 30.5%. It is most decisive for large bias/dropout and less decisive for small bias, drift, and noise.**
4. **M(t) is normalized only with clean validation data. The uncertainty cases spent 8.2% of their time in HIGH, causing shorter horizons, larger move penalties, and smaller voltage steps without a hard fault class.**
5. **Across the five comparison scenarios, mean RMSE was 13.458 for adaptive, 12.345 for plain MPC, and 12.271 for sensor-only MPC. Adaptation is retained as a safety response even where it does not improve RMSE.**
6. **Applied-command voltage/rate violations: 0; optimizer failures handled by PI fallback: 0; mean successful MPC call time: 151.5 ms on this CPU. Speed, NaN/Inf, slew, and optimizer guards were active.**
7. **The LSTM was trained open-loop and even the selected 200 ms prediction horizon is short relative to motor settling. Sensor residuals can still react to plant mismatch, and SLSQP CPU time exceeds the 50 ms controller interval in some runs. The next step is timing-aware deployment validation or a faster solver/distilled predictor, not another diagnostic classifier.**

Phase 5 checks passed: 27 controller/scenario runs, 11 artifacts, no applied constraint violations.
